#**Statistical Analysis of Differences Between Groups**

###Registered vs Non-Registered Users — Sales Comparison

In [ ]:

df['date'] = pd.to_datetime(df['date'])

df['is_registered'] = df['account_id'].notna()

daily_sales = df.groupby(['date','is_registered'])['price'].sum().unstack()

registered_sales = daily_sales[True].dropna()
non_registered_sales = daily_sales[False].dropna()

stat_reg, p_reg = shapiro(registered_sales)
stat_nonreg, p_nonreg = shapiro(non_registered_sales)

print("Registered p-value:", p_reg)
print("Non-registered p-value:", p_nonreg)


if p_reg > 0.05 and p_nonreg > 0.05:
    stat, p_value = ttest_ind(registered_sales, non_registered_sales)
    test_used = "Independent t-test"
else:
    stat, p_value = mannwhitneyu(registered_sales, non_registered_sales)
    test_used = "Mann-Whitney U test"

print("Test used:", test_used)
print("p-value:", p_value)



Registered p-value: 0.007295139880540744
Non-registered p-value: 0.0012184604855253293
Test used: Mann-Whitney U test
p-value: 3.8805185465235906e-26


###Sessions by Traffic Channels

In [ ]:

channel_sessions = (
    df.groupby(['date','traffic_channel'])['ga_session_id']
    .nunique()
    .unstack()
)

samples = [channel_sessions[col].dropna() for col in channel_sessions.columns]


normality = [shapiro(sample)[1] > 0.05 for sample in samples]

if all(normality):
    stat, p_value = f_oneway(*samples)
    test_used = "ANOVA"
else:
    stat, p_value = kruskal(*samples)
    test_used = "Kruskal-Wallis"

print("Test used:", test_used)
print("p-value:", p_value)


Test used: Kruskal-Wallis
p-value: 1.397036102599359e-78


###Difference in Share of Organic Sessions (Europe vs Americas)

In [ ]:
regions = df[df['continent'].isin(['Europe','Americas'])]

total_sessions = regions.groupby('continent')['ga_session_id'].nunique()

organic_sessions = (
    regions[regions['traffic_channel'] == 'Organic Search']
    .groupby('continent')['ga_session_id']
    .nunique()
)

count = [organic_sessions['Europe'], organic_sessions['Americas']]
nobs = [total_sessions['Europe'], total_sessions['Americas']]


stat, p_value = proportions_ztest(count, nobs)

print("Z-statistic:", stat)
print("p-value:", p_value)


Z-statistic: 0.28951412926103953
p-value: 0.7721879690501752


###Levene’s Test

In [ ]:
stat, p_value = levene(registered_sales, non_registered_sales)

print("Levene test p-value:", p_value)


Levene test p-value: 3.881339271971661e-15


###Chi-Square Test (χ²)

In [ ]:
contingency_table = pd.crosstab(df['continent'], df['traffic_channel'])

stat, p_value, dof, expected = chi2_contingency(contingency_table)

print("Chi-square p-value:", p_value)
if p_value < 0.05:
    print('there is a relationship between variables')
else:
    print('independent variables')

Chi-square p-value: 0.43954438481889735
independent variables
